# Лабораторная работа 8
**Тема.** 

**Задача 1.**
$$
J_3 | \space | C_{max}
$$ 


Эвристический: Shifting Bottleneck

In [1]:
tasks = {
    'J1': [(1, 6), (2, 8), (4, 5)],
    'J2': [(2, 4), (3, 4), (4, 3)],
    'J3': [(4, 8), (2, 6), (1, 4)],
    'J4': [(2, 5), (3, 10), (4, 15)]
}

In [2]:
def solve_jm_spt(jobs):
    all_machines = set()
    for route in jobs.values():
        for m_id, p in route:
            all_machines.add(m_id)
            
    job_step = {j_id: 0 for j_id in jobs}
    machine_free = {m_id: 0 for m_id in all_machines}
    job_ready = {j_id: 0 for j_id in jobs}
    
    finished = []
    num_jobs = len(jobs)
    
    while len(finished) < num_jobs:
        candidates = []
        for j_id, step_idx in job_step.items():
            if step_idx < len(jobs[j_id]):
                m_id, p = jobs[j_id][step_idx]
                start_time = max(machine_free[m_id], job_ready[j_id])
                candidates.append((start_time, p, j_id, m_id))
        
        if not candidates:
            break
            
        candidates.sort(key=lambda x: (x[0], x[1]))
        
        start, p, j_id, m_id = candidates[0]
        
        end_time = start + p
        machine_free[m_id] = end_time
        job_ready[j_id] = end_time
        job_step[j_id] += 1
        
        if job_step[j_id] == len(jobs[j_id]):
            finished.append((j_id, end_time))
            
    return max(f[1] for f in finished)


c_max = solve_jm_spt(tasks)
print(f"Результат для J4 | | C_max (по правилу SPT): {c_max}")

Результат для J4 | | C_max (по правилу SPT): 39


Метод ветвей и границ

In [3]:
import copy

def get_critical_path(jobs, machine_orders):
    job_step = {j: 0 for j in jobs}
    m_idx = {m: 0 for m in machine_orders}
    m_free = {m: 0 for m in machine_orders}
    j_ready = {j: 0 for j in jobs}
    
    scheduled_ops = 0
    total_ops = sum(len(v) for v in jobs.values())
    
    while scheduled_ops < total_ops:
        progress = False
        for m_id, order in machine_orders.items():
            if m_idx[m_id] < len(order):
                j_id = order[m_idx[m_id]]
                curr_step = job_step[j_id]
                target_m, p = jobs[j_id][curr_step]
                
                if target_m == m_id and j_ready[j_id] <= m_free[m_id]:
                    # Выполняем
                    start = max(m_free[m_id], j_ready[j_id])
                    end = start + p
                    m_free[m_id] = end
                    j_ready[j_id] = end
                    m_idx[m_id] += 1
                    job_step[j_id] += 1
                    scheduled_ops += 1
                    progress = True
        if not progress: break
    return max(m_free.values()) if scheduled_ops == total_ops else float('inf')

def solve_jm_bnb(jobs):
    all_m = set(m for route in jobs.values() for m, p in route)
    all_j = list(jobs.keys())
    
    best_c = float('inf')
    best_orders = {}

    from itertools import permutations
    possible_permu = list(permutations(all_j))
    
    import itertools
    for orders in itertools.product(possible_permu, repeat=len(all_m)):
        m_orders = {m_id: orders[i] for i, m_id in enumerate(all_m)}
        c = get_critical_path(jobs, m_orders)
        if c < best_c:
            best_c = c
            best_orders = m_orders
            
    return best_c, best_orders

def print_schedule_table(jobs, result_cmax):
    job_step = {j: 0 for j in jobs}
    m_free = {m: 0 for m in [1, 2, 3, 4]}
    j_ready = {j: 0 for j in jobs}
    
    print(f"{'Работа':<7} | {'Станок':<7} | {'Начало':<7} | {'Конец':<7} | {'Длительность'}")
    print("-" * 55)

    completed = 0
    total = sum(len(v) for v in jobs.values())
    
    history = []
    while completed < total:
        candidates = []
        for j_id, step_idx in job_step.items():
            if step_idx < len(jobs[j_id]):
                m_id, p = jobs[j_id][step_idx]
                start = max(m_free[m_id], j_ready[j_id])
                candidates.append((start, p, j_id, m_id))
        
        candidates.sort(key=lambda x: (x[0], x[1]))
        start, p, j_id, m_id = candidates[0]
        
        end = start + p
        print(f"{j_id:<7} | M{m_id:<6} | {start:<7} | {end:<7} | {p}")
        
        m_free[m_id] = end
        j_ready[j_id] = end
        job_step[j_id] += 1
        completed += 1

    print("-" * 55)
    print(f"Итоговый C_max: {result_cmax}")

print_schedule_table(tasks, c_max)

Работа  | Станок  | Начало  | Конец   | Длительность
-------------------------------------------------------
J2      | M2      | 0       | 4       | 4
J1      | M1      | 0       | 6       | 6
J3      | M4      | 0       | 8       | 8
J2      | M3      | 4       | 8       | 4
J4      | M2      | 4       | 9       | 5
J2      | M4      | 8       | 11      | 3
J3      | M2      | 9       | 15      | 6
J4      | M3      | 9       | 19      | 10
J3      | M1      | 15      | 19      | 4
J1      | M2      | 15      | 23      | 8
J4      | M4      | 19      | 34      | 15
J1      | M4      | 34      | 39      | 5
-------------------------------------------------------
Итоговый C_max: 39
